# TỰ ĐỘNG NHẬN DIỆN VÀ ĐỌC ĐA ĐỊNH DẠNG FILE BÁO CÁO TMĐT BẨN

In [1]:
import os
import sys
import joblib
import pandas as pd
import numpy as np

In [2]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
MESSY_DIR = os.path.join(ROOT, 'data', 'samples_messy')
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
MODEL_DIR = os.path.join(ROOT, 'models')

In [3]:
#Xem danh sách 10 file xuất thô mô phỏng các sàn TMĐT
files = sorted(os.listdir(MESSY_DIR))
print('Số lượng file test:', len(files))
files

Số lượng file test: 10


['01_shopee_clean.xlsx',
 '02_tiki_semicolon_bom.csv',
 '03_lazada_english.xlsx',
 '04_tiktok_title_offset.xlsx',
 '05_pos_legacy.xls',
 '06_multi_sheet_report.xlsx',
 '07_utf16_tab_export.csv',
 '08_two_text_cols.xlsx',
 '09_merged_cells_pagination.xlsx',
 '10_html_table_as_xls.xls']

## 1. ĐỌC ĐA ĐỊNH DẠNG FILE BÁO CÁO

In [4]:
#Nạp bộ đọc và nhận diện cấu trúc bảng Excel/CSV dùng chung từ src.excel_ingestion
from src.excel_ingestion import (
    read_any_file, pick_best_sheet, detect_header_row,
    detect_text_columns, detect_star_column, build_clean_table, mask_tabular_pii
)

In [5]:
#Thử nghiệm đọc file HTML đội lốt XLS bằng read_any_file
sheets_10, meta_10 = read_any_file(os.path.join(MESSY_DIR, '10_html_table_as_xls.xls'))
tempt_df = sheets_10['Sheet1']
print('Định dạng nhận diện:', meta_10['format'])
tempt_df.head(2)

Định dạng nhận diện: html_xls


,Mã ĐH,Người Mua,Nội Dung Đánh Giá,Đánh Giá Sao,Thời Gian
0,ORD933552,Hoàng Thảo Vy,"Sản phẩm tốt, nhưng pin không trâu như mình ng...",3,2026-10-09
1,ORD479410,Đỗ Minh Đức,Hàng chất lượng sai rất ok cho thế giới đi đôn...,5,2026-10-15


In [6]:
tempt_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Mã ĐH              25 non-null     str  
 1   Người Mua          25 non-null     str  
 2   Nội Dung Đánh Giá  25 non-null     str  
 3   Đánh Giá Sao       25 non-null     int64
 4   Thời Gian          25 non-null     str  
dtypes: int64(1), str(4)
memory usage: 6.8 KB


## 2. TỰ ĐỘNG CHỌN SHEET CHỨA NHẬN XÉT

In [7]:
#Kiểm tra tự chọn sheet trên file nhiều sheet 06_multi_sheet_report.xlsx
sheets_06, meta_06 = read_any_file(os.path.join(MESSY_DIR, '06_multi_sheet_report.xlsx'))
best_sname, df_best = pick_best_sheet(sheets_06)
print('Sheet được chọn tự động:', best_sname)
df_best.head(2)

Sheet được chọn tự động: Danh_Gia_Chi_Tiet


,0,1,2,3,4
0,Mã Đơn,Khách Hàng,Nội dung phản hồi,Đánh giá sao,Chi nhánh
1,ORD782784,Hoàng Thảo Vy,"Sản phẩm tốt, nhưng pin không trâu như mình ng...",3,Hà Nội


## 3. TỰ ĐỘNG DÒ DÒNG TIÊU ĐỀ VÀ CỘT ĐÁNH GIÁ

In [8]:
#Thử dò dòng tiêu đề thật khi bảng bị đẩy lệch bởi banner trên file 04
sheets_04, meta_04 = read_any_file(os.path.join(MESSY_DIR, '04_tiktok_title_offset.xlsx'))
sname_04, raw_04 = pick_best_sheet(sheets_04)
hdr_04 = detect_header_row(raw_04)
print('Dòng tiêu đề nhận diện được:', hdr_04)

Dòng tiêu đề nhận diện được: 4


In [9]:
#Kiểm tra dò cột văn bản và cột số sao trên file 04 sau khi làm sạch bảng
clean_tab_04 = build_clean_table(raw_04, hdr_04, meta=meta_04, sheet_name=sname_04)
detect_text_columns(clean_tab_04), detect_star_column(clean_tab_04)

(['Ý kiến đánh giá', 'review_text'], 'Số sao')

## 4. XỬ LÝ LẶP TIÊU ĐỀ, Ô GỘP VÀ ĐA CỘT VĂN BẢN

In [10]:
#Thử nghiệm gộp 2 cột nhận xét thành review_text trên file 08_two_text_cols.xlsx
sheets_08, meta_08 = read_any_file(os.path.join(MESSY_DIR, '08_two_text_cols.xlsx'))
sname_08, raw_08 = pick_best_sheet(sheets_08)
clean_tab_08 = build_clean_table(raw_08, detect_header_row(raw_08), meta=meta_08, sheet_name=sname_08)
clean_tab_08[['Nhận xét về máy', 'Ý kiến đóng gói giao hàng', 'review_text']].head(3)

,Nhận xét về máy,Ý kiến đóng gói giao hàng,review_text
0,"Sản phẩm tốt, nhưng pin không trâu như mình ng...",Hộp đóng gói cẩn thận 3 lớp bóng khí bọc quanh...,"Sản phẩm tốt, nhưng pin không trâu như mình ng..."
1,Hàng chất lượng sai rất ok cho thế giới đi đôn...,Giao hơi chậm 1 ngày so với dự kiến nhưng ship...,Hàng chất lượng sai rất ok cho thế giới đi đôn...
2,E mới mưa đc 2 tháng. Mà e mới cập nhật phiên ...,Hộp đóng gói cẩn thận 3 lớp bóng khí bọc quanh...,E mới mưa đc 2 tháng. Mà e mới cập nhật phiên ...


In [11]:
clean_tab_08.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Mã đơn                     25 non-null     str   
 1   Khách hàng                 25 non-null     str   
 2   Nhận xét về máy            25 non-null     str   
 3   Ý kiến đóng gói giao hàng  25 non-null     str   
 4   Sao                        25 non-null     object
 5   Ngày                       25 non-null     str   
 6   review_text                25 non-null     str   
dtypes: object(1), str(6)
memory usage: 15.8+ KB


In [12]:
#Truyền meta và sheet_name để chỉ unmerge đúng vùng ô gộp Nhóm sản phẩm và lọc tiêu đề lặp giữa trang
sheets_09, meta_09 = read_any_file(os.path.join(MESSY_DIR, '09_merged_cells_pagination.xlsx'))
sname_09, raw_09 = pick_best_sheet(sheets_09)
clean_tab_09 = build_clean_table(raw_09, detect_header_row(raw_09), meta=meta_09, sheet_name=sname_09)
clean_tab_09.head(3)

,STT,Nhóm sản phẩm,Khách hàng,Phản hồi sản phẩm,Số sao đánh giá,Ghi chú,review_text
0,1,Điện thoại Flagship,Hoàng Thảo Vy,"Sản phẩm tốt, nhưng pin không trâu như mình ng...",3,Đã xử lý,"Sản phẩm tốt, nhưng pin không trâu như mình ng..."
1,2,Điện thoại Flagship,Trần Thị Bích,Hàng chất lượng sai rất ok cho thế giới đi đôn...,5,Chưa xử lý,Hàng chất lượng sai rất ok cho thế giới đi đôn...
2,3,Điện thoại Flagship,Trần Thị Bích,E mới mưa đc 2 tháng. Mà e mới cập nhật phiên ...,4,Chưa xử lý,E mới mưa đc 2 tháng. Mà e mới cập nhật phiên ...


In [13]:
clean_tab_09.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   STT                25 non-null     object
 1   Nhóm sản phẩm      25 non-null     str   
 2   Khách hàng         25 non-null     str   
 3   Phản hồi sản phẩm  25 non-null     str   
 4   Số sao đánh giá    25 non-null     object
 5   Ghi chú            25 non-null     str   
 6   review_text        25 non-null     str   
dtypes: object(2), str(5)
memory usage: 12.6+ KB


In [14]:
#Xác nhận file 09 còn đúng 25 dòng, 0 NaN ở Nhóm sản phẩm và không bị ffill bừa
print('Số dòng:', len(clean_tab_09), '| NaN Nhóm sản phẩm:', clean_tab_09['Nhóm sản phẩm'].isna().sum())
clean_tab_09['Nhóm sản phẩm'].value_counts()

Số dòng: 25 | NaN Nhóm sản phẩm: 0


Nhóm sản phẩm
Điện thoại Flagship     5
Điện thoại Tầm trung    5
Phụ kiện & Sạc nhanh    5
Điện thoại Phổ thông    5
Dịch vụ & Hậu mãi       5
Name: count, dtype: int64

In [15]:
#Kiểm thử nhanh giữ nguyên ô trống ở cột SĐT/Số sao vì chỉ unmerge đúng merged_ranges
tempt_raw = pd.DataFrame([['Nhóm', 'SĐT', 'Nhận xét', 'Số sao'], ['A', '0901234567', 'Sản phẩm tốt lắm ạ', 5], [None, None, 'Pin hơi yếu một chút', None]])
tempt_res = build_clean_table(tempt_raw, header_row=0, meta={'merged_ranges': {'S1': [(2, 3, 1, 1)]}}, sheet_name='S1')
assert tempt_res['Nhóm'].tolist() == ['A', 'A'] and pd.isna(tempt_res.loc[1, 'SĐT']) and pd.isna(tempt_res.loc[1, 'Số sao'])
tempt_res

,Nhóm,SĐT,Nhận xét,Số sao,review_text
0,A,0901234567,Sản phẩm tốt lắm ạ,5,Sản phẩm tốt lắm ạ
1,A,NaN,Pin hơi yếu một chút,None,Pin hơi yếu một chút


## 5. CHẠY KIỂM ĐỊNH BENCHMARK TRÊN TOÀN BỘ 10 FILE BẨN

In [16]:
#Thiết lập nhãn cột chuẩn cho 10 file xuất thô
truth = {
    '01_shopee_clean.xlsx': {'text': ['Đánh giá của khách'], 'star': 'Số sao (1-5)'}, '02_tiki_semicolon_bom.csv': {'text': ['Review_Content'], 'star': 'Rating_Star'},
    '03_lazada_english.xlsx': {'text': ['Comment / Review'], 'star': 'Rating'}, '04_tiktok_title_offset.xlsx': {'text': ['Ý kiến đánh giá'], 'star': 'Số sao'},
    '05_pos_legacy.xls': {'text': ['Nhan_Xet'], 'star': 'Diem_Sao'}, '06_multi_sheet_report.xlsx': {'text': ['Nội dung phản hồi'], 'star': 'Đánh giá sao'},
    '07_utf16_tab_export.csv': {'text': ['Review_Text'], 'star': 'Stars'}, '08_two_text_cols.xlsx': {'text': ['Nhận xét về máy', 'Ý kiến đóng gói giao hàng'], 'star': 'Sao'},
    '09_merged_cells_pagination.xlsx': {'text': ['Phản hồi sản phẩm'], 'star': 'Số sao đánh giá'}, '10_html_table_as_xls.xls': {'text': ['Nội Dung Đánh Giá'], 'star': 'Đánh Giá Sao'}
}

In [17]:
#Chạy kiểm định tự động trên cả 10 file bẩn
results = []
for fname, expected in sorted(truth.items()):
    sheets, meta = read_any_file(os.path.join(MESSY_DIR, fname))
    best_sname, best_df = pick_best_sheet(sheets)
    tab = build_clean_table(best_df, detect_header_row(best_df), meta=meta, sheet_name=best_sname)
    det_txt, det_star = detect_text_columns(tab), detect_star_column(tab)
    ok = (len(tab) == 25) and all(t in det_txt for t in expected['text']) and (det_star == expected['star']) and bool((tab['review_text'].str.len() > 0).all())
    results.append({'file_name': fname, 'rows': len(tab), 'format': meta['format'], 'detected_text': ', '.join(det_txt), 'detected_star': str(det_star), 'pass_status': 'PASS' if ok else 'FAIL'})
df_benchmark = pd.DataFrame(results)

In [18]:
print('Tỷ lệ nhận diện thành công: ', (df_benchmark['pass_status'] == 'PASS').sum(), '/', len(df_benchmark))
df_benchmark

Tỷ lệ nhận diện thành công:  10 / 10


,file_name,rows,format,detected_text,detected_star,pass_status
0,01_shopee_clean.xlsx,25,xlsx,"Đánh giá của khách, review_text",Số sao (1-5),PASS
1,02_tiki_semicolon_bom.csv,25,csv,"Review_Content, review_text",Rating_Star,PASS
2,03_lazada_english.xlsx,25,xlsx,"Comment / Review, review_text",Rating,PASS
3,04_tiktok_title_offset.xlsx,25,xlsx,"Ý kiến đánh giá, review_text",Số sao,PASS
4,05_pos_legacy.xls,25,biff_xls,"Nhan_Xet, review_text",Diem_Sao,PASS
5,06_multi_sheet_report.xlsx,25,xlsx,"Nội dung phản hồi, review_text",Đánh giá sao,PASS
6,07_utf16_tab_export.csv,25,csv,"Review_Text, review_text",Stars,PASS
7,08_two_text_cols.xlsx,25,xlsx,"Nhận xét về máy, Ý kiến đóng gói giao hàng, re...",Sao,PASS
8,09_merged_cells_pagination.xlsx,25,xlsx,"Phản hồi sản phẩm, review_text",Số sao đánh giá,PASS
9,10_html_table_as_xls.xls,25,html_xls,"Nội Dung Đánh Giá, review_text",Đánh Giá Sao,PASS


## 6. CHUẨN HÓA VĂN BẢN, ẨN DANH PII VÀ TÁCH TỪ PYVI TRƯỚC DỰ ĐOÁN

In [19]:
#Nạp bộ làm sạch văn bản, lọc câu rác và tách từ pyvi dùng chung từ src.text_cleaner
from src.text_cleaner import clean_text, check_no_content, segment_words

In [20]:
#Thử nghiệm làm sạch, lọc nội dung và tách từ trên 1 câu mẫu chứa SĐT và mã đơn
tempt = 'Đơn ORD967144 của sđt 0779887766: TGDĐ đt dùng rất mượttt nhưng pin tụt ko dùng đk lâu'
tempt_clean = clean_text(tempt, lowercase=False)
tempt_seg = segment_words(tempt_clean)
tempt_clean, tempt_seg, check_no_content(tempt_clean), check_no_content('cho 5 sao nhận xu')

('Đơn [ORDER_ID] của sđt [PHONE] : Thế Giới Di Động điện thoại dùng rất mượt nhưng pin tụt không dùng được lâu',
 'Đơn [ORDER_ID] của sđt [PHONE] : Thế_Giới_Di_Động điện_thoại dùng rất mượt nhưng pin tụt không dùng được lâu',
 False,
 True)

## 7. DỰ ĐOÁN HÀNG LOẠT VÀ XUẤT FILE SAMPLE_EXCEL_PREDICTED.XLSX

In [21]:
#Nạp mô hình Baseline TF-IDF + LinearSVC cho bước chạy thử nghiệm (Task 7 sẽ chuyển sang PhoBertMultiTask sau khi huấn luyện xong trên Colab)
tfidf = joblib.load(os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))
svc_sent = joblib.load(os.path.join(MODEL_DIR, 'svc_sentiment.pkl'))
svc_asp = joblib.load(os.path.join(MODEL_DIR, 'svc_aspect.pkl'))

In [22]:
#Đọc và chuẩn hóa bảng dữ liệu từ file 01_shopee_clean.xlsx
sheets_sample, meta_sample = read_any_file(os.path.join(MESSY_DIR, '01_shopee_clean.xlsx'))
sname_sample, raw_sample = pick_best_sheet(sheets_sample)
df_sample = build_clean_table(raw_sample, detect_header_row(raw_sample), meta=meta_sample, sheet_name=sname_sample)
txt_cols = [c for c in detect_text_columns(df_sample) if c != 'review_text']
txt_col = txt_cols[0] if txt_cols else 'review_text'
star_col = detect_star_column(df_sample)
df_sample.shape, txt_col, star_col

((25, 7), 'Đánh giá của khách', 'Số sao (1-5)')

In [23]:
#Ẩn danh PII trên các cột bảng (Tên người mua, SĐT, Mã đơn) và làm sạch che PII trên cột văn bản gốc
df_sample = mask_tabular_pii(df_sample, text_clean_fn=lambda s: clean_text(s, lowercase=False))
df_sample.head(3)

,Mã đơn hàng,Tên người mua,Số điện thoại,Đánh giá của khách,Số sao (1-5),Thời gian,review_text
0,[ORDER_ID],[PERSON],[PHONE],"Sản phẩm tốt, nhưng pin không trâu như mình ng...",3,2026-10-14,"Sản phẩm tốt, nhưng pin không trâu như mình ng..."
1,[ORDER_ID],[PERSON],[PHONE],Hàng chất lượng sai rất ok cho thế giới đi đôn...,5,2026-10-24,Hàng chất lượng sai rất ok cho thế giới đi đôn...
2,[ORDER_ID],[PERSON],[PHONE],E mới mưa được 2 tháng. Mà e mới cập nhật phiê...,4,2026-10-03,E mới mưa được 2 tháng. Mà e mới cập nhật phiê...


In [24]:
#Tạo sentence_clean, tách từ pyvi cho sentence_seg và đánh dấu câu rỗng/săn xu bằng check_no_content
df_sample['sentence_clean'] = df_sample['review_text']
df_sample['sentence_seg'] = df_sample['sentence_clean'].apply(segment_words)
df_sample['is_no_content'] = df_sample['sentence_clean'].apply(check_no_content)
df_sample[[txt_col, 'sentence_clean', 'sentence_seg', 'is_no_content']].head(3)

,Đánh giá của khách,sentence_clean,sentence_seg,is_no_content
0,"Sản phẩm tốt, nhưng pin không trâu như mình ng...","Sản phẩm tốt, nhưng pin không trâu như mình ng...","Sản_phẩm tốt , nhưng pin không trâu như mình n...",False
1,Hàng chất lượng sai rất ok cho thế giới đi đôn...,Hàng chất lượng sai rất ok cho thế giới đi đôn...,Hàng chất_lượng sai rất ok cho thế_giới đi đôn...,False
2,E mới mưa được 2 tháng. Mà e mới cập nhật phiê...,E mới mưa được 2 tháng. Mà e mới cập nhật phiê...,E mới mưa được 2 tháng . Mà_e mới cập_nhật phi...,False


In [25]:
#Đưa văn bản đã tách từ sentence_seg vào tfidf.transform để dự đoán cảm xúc và khía cạnh
X_sample = tfidf.transform(df_sample['sentence_seg'])
p_sent = svc_sent.predict(X_sample)
p_asp = svc_asp.predict(X_sample)
X_sample.shape, X_sample.nnz

((25, 12000), 1029)

In [26]:
#Ánh xạ nhãn dự đoán và gắn cờ kiểm duyệt khi mâu thuẫn số sao (an toàn khi thiếu cột sao)
sent_map = {0: 'Tiêu cực', 1: 'Trung lập', 2: 'Tích cực'}
asp_map = {0: 'Hiệu năng & Cấu hình', 1: 'Màn hình & Camera', 2: 'Thời lượng Pin & Sạc', 3: 'Giá cả & Hậu mãi', 4: 'Thiết kế & Tổng quan'}
df_sample['Du_Doan_Cam_Xuc'] = [sent_map[s] for s in p_sent]
df_sample['Du_Doan_Khia_Canh'] = [asp_map[a] for a in p_asp]
star_num = pd.to_numeric(df_sample[star_col], errors='coerce') if star_col is not None else None
star_contra = (((star_num <= 2) & (p_sent == 2)) | ((star_num >= 4) & (p_sent == 0))) if star_col is not None else False
df_sample['Can_Kiem_Duyet'] = star_contra

In [27]:
df_sample['Du_Doan_Cam_Xuc'].value_counts()

Du_Doan_Cam_Xuc
Tích cực     14
Tiêu cực     10
Trung lập     1
Name: count, dtype: int64

In [28]:
df_sample['Du_Doan_Khia_Canh'].value_counts()

Du_Doan_Khia_Canh
Hiệu năng & Cấu hình    10
Giá cả & Hậu mãi         6
Thời lượng Pin & Sạc     3
Thiết kế & Tổng quan     3
Màn hình & Camera        3
Name: count, dtype: int64

In [29]:
df_sample['is_no_content'].value_counts()

is_no_content
False    25
Name: count, dtype: int64

In [30]:
cols_show = [c for c in ['Mã đơn hàng', 'Tên người mua', 'Số điện thoại', txt_col, 'sentence_clean', star_col, 'Du_Doan_Cam_Xuc', 'Du_Doan_Khia_Canh', 'Can_Kiem_Duyet'] if c in df_sample.columns]
df_sample[cols_show].head(5)

,Mã đơn hàng,Tên người mua,Số điện thoại,Đánh giá của khách,sentence_clean,Số sao (1-5),Du_Doan_Cam_Xuc,Du_Doan_Khia_Canh,Can_Kiem_Duyet
0,[ORDER_ID],[PERSON],[PHONE],"Sản phẩm tốt, nhưng pin không trâu như mình ng...","Sản phẩm tốt, nhưng pin không trâu như mình ng...",3,Trung lập,Thời lượng Pin & Sạc,False
1,[ORDER_ID],[PERSON],[PHONE],Hàng chất lượng sai rất ok cho thế giới đi đôn...,Hàng chất lượng sai rất ok cho thế giới đi đôn...,5,Tích cực,Giá cả & Hậu mãi,False
2,[ORDER_ID],[PERSON],[PHONE],E mới mưa được 2 tháng. Mà e mới cập nhật phiê...,E mới mưa được 2 tháng. Mà e mới cập nhật phiê...,4,Tiêu cực,Thời lượng Pin & Sạc,True
3,[ORDER_ID],[PERSON],[PHONE],Mới mua hôm nay giao hàng ngay hôm nay rất nha...,Mới mua hôm nay giao hàng ngay hôm nay rất nha...,5,Tích cực,Giá cả & Hậu mãi,False
4,[ORDER_ID],[PERSON],[PHONE],Sao máy lúc nào cũng nóng vậy nhỉ mình có để t...,Sao máy lúc nào cũng nóng vậy nhỉ mình có để t...,3,Tiêu cực,Hiệu năng & Cấu hình,False


In [31]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Mã đơn hàng         25 non-null     object
 1   Tên người mua       25 non-null     object
 2   Số điện thoại       25 non-null     object
 3   Đánh giá của khách  25 non-null     str   
 4   Số sao (1-5)        25 non-null     object
 5   Thời gian           25 non-null     str   
 6   review_text         25 non-null     str   
 7   sentence_clean      25 non-null     str   
 8   sentence_seg        25 non-null     str   
 9   is_no_content       25 non-null     bool  
 10  Du_Doan_Cam_Xuc     25 non-null     str   
 11  Du_Doan_Khia_Canh   25 non-null     str   
 12  Can_Kiem_Duyet      25 non-null     bool  
dtypes: bool(2), object(4), str(7)
memory usage: 23.3+ KB


In [32]:
#Xuất file báo cáo mẫu đã làm sạch và ẩn danh PII hoàn toàn ra thư mục processed
out_pred_excel = os.path.join(PROC_DIR, 'sample_excel_predicted.xlsx')
df_sample.to_excel(out_pred_excel, index=False, engine='openpyxl')
print('Đã xuất thành công file báo cáo kết quả:', out_pred_excel)

Đã xuất thành công file báo cáo kết quả: D:\E-commerce\data\processed\sample_excel_predicted.xlsx


In [33]:
df_check = pd.read_excel(out_pred_excel)
df_check.shape

(25, 13)